# Hybrid Mode Comparison

This notebook explains the direct per-MAID hybrid family. In this version, batch computes the static candidate list for each MAID, and the online path applies either live delivery checks, bitmap gating, or bitmap gating plus taxonomy rechecks before reranking.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from data.common import read_jsonl

dataset_dir = Path('../data/generated/synthetic')
reports_dir = Path('../reports/generated')
metadata = json.loads((dataset_dir / 'metadata.json').read_text(encoding='utf-8'))
maids = read_jsonl(dataset_dir / 'maids.jsonl')
user_candidates = read_jsonl(dataset_dir / 'user_candidates.jsonl')
campaigns = read_jsonl(dataset_dir / 'campaigns.jsonl')
evaluation = json.loads((reports_dir / 'evaluation.json').read_text(encoding='utf-8'))
benchmark = json.loads((reports_dir / 'hybrid_benchmark.json').read_text(encoding='utf-8'))
shadow = json.loads((reports_dir / 'hybrid_shadow_smoke.json').read_text(encoding='utf-8'))
metadata


## 1. Batch Output

The batch process now evaluates each campaign's static targeting against each MAID and writes the resulting candidate campaigns directly onto that MAID.

That means `aud:{maid_id}` is no longer a list of audience buckets. It is the actual precomputed candidate campaign list for that MAID.


In [ ]:
maid = maids[0]
maid_candidates = user_candidates[0]
campaign_lookup = {row["campaign_id"]: row for row in campaigns}
sample_campaigns = [campaign_lookup[campaign_id] for campaign_id in maid_candidates["candidate_ids"][:3]]
display({"maid": maid, "precomputed_candidates": maid_candidates, "sample_campaigns": sample_campaigns})


## 2. Redis Keys

The online keys used by the direct-list hybrid mode are:

- `identity:{token}` -> `maid_id`
- `maid:{maid_id}` -> full MAID profile hash used by `full_realtime`
- `maid_hot:{maid_id}` -> compact scoring profile used by precomputed modes
- `aud:{maid_id}` -> precomputed campaign IDs for that MAID
- `campaign:{campaign_id}` -> campaign metadata hash
- `campaign_state:{campaign_id}` -> mutable delivery state for the non-bitmap hybrid path
- `fcap:{maid_id}` -> per-user delivery counters as a hash
- `bm:servable` -> global active+pacing+budget eligibility bitmap


In [ ]:
maid_id = maid["user_id"]
campaign_id = maid_candidates["candidate_ids"][0]
commands = [
    f"GET identity:{maid['identity_tokens'][0]}",
    f"HGETALL maid_hot:{maid_id}",
    f"GET aud:{maid_id}",
    f"HGETALL campaign:{campaign_id}",
    f"HGETALL campaign_state:{campaign_id}",
    f"HMGET fcap:{maid_id} {campaign_id}",
]
commands


## 3. Mode Comparison

Because batch now does the full static targeting selection, the precomputed modes match the full real-time baseline exactly on the synthetic dataset.


In [ ]:
offline = pd.DataFrame(evaluation["synthetic_modes"]["modes"]).T
offline[[
    "ndcg_at_k",
    "candidate_generation_recall",
    "eligible_recall",
    "top_result_jaccard_vs_full_realtime",
    "candidate_count",
    "eligible_count",
]].sort_index()


## 4. Live Serial Benchmarks

The key latency improvement from this design is that candidate generation becomes a single `GET aud:{maid_id}` or a single server-side bitmap-gated lookup instead of an audience indirection plus multiple candidate-list lookups. The reranker still gets user-level scoring signals, but it reads them from the small `maid_hot:{maid_id}` hash rather than the full MAID profile. The newest `hybrid_bitmap_taxonomy` mode keeps that cheap retrieval path and adds back the per-ad `taxonomy_filter` AND/OR/NOT check in app memory.


In [ ]:
live = pd.DataFrame(benchmark["loadtests"]).T
live[[
    "handler_avg_latency_ms",
    "handler_p95_latency_ms",
    "candidate_generation_avg_latency_ms",
    "campaign_fetch_avg_latency_ms",
    "avg_candidate_count",
    "avg_eligible_count",
    "avg_redis_round_trips",
]].sort_index()


## 5. Shadow Execution

When hybrid runs with the other two modes in shadow, the top-result overlap is now perfect on the synthetic dataset because all three modes are operating on the same static candidate truth.


In [ ]:
shadow


## 6. Tradeoffs

- `full_realtime` stays as the correctness baseline, but it still pays to materialize every campaign on the hot path.
- `precomputed_segment`, `hybrid_precompute_plus_realtime`, `hybrid_bitmap_gating`, and `hybrid_bitmap_taxonomy` are all driven by direct per-MAID candidate lists.
- `hybrid_precompute_plus_realtime` keeps the full live exact-check stage, including `taxonomy_filter`, so it remains the correctness-preserving low-latency path.
- `hybrid_bitmap_gating` is the low-fanout experiment that skips taxonomy.
- `hybrid_bitmap_taxonomy` is the low-fanout experiment that adds taxonomy checks back without restoring campaign-state fanout.
- Versioning is now batch provenance only. Because the local workflow fully reloads Redis, there is no online stale-key cleanup problem today.
